In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# ==============================================================================
# STEP 1: HIGH-FIDELITY SIMULATED DATA SETUP
# ==============================================================================

# --- 1. Master Fund Data (40 Schemes) ---
fund_names = [f"Alpha Quantum Fund {i}" for i in range(1, 16)] + \
             [f"Beta Steady Growth {i}" for i in range(1, 16)] + \
             [f"Gamma Defensive Allocation {i}" for i in range(1, 11)]

risk_grades = ['High'] * 15 + ['Moderate'] * 15 + ['Low'] * 10
fund_types = ['Equity'] * 15 + ['Hybrid'] * 15 + ['Debt'] * 10
funds_df = pd.DataFrame({'fund_name': fund_names, 'risk_grade': risk_grades, 'fund_type': fund_types})

# --- 2. Daily Returns Generation (2 Years) ---
dates = pd.date_range(start="2024-01-01", end="2026-01-01", freq="B")
returns_data = {}
for i, fund in enumerate(fund_names):
    if risk_grades[i] == 'High':
        mean, std = 0.0006, 0.015
    elif risk_grades[i] == 'Moderate':
        mean, std = 0.0004, 0.009
    else:
        mean, std = 0.0002, 0.004
    returns_data[fund] = np.random.normal(mean, std, len(dates))

returns_df = pd.DataFrame(returns_data, index=dates)

# --- 3. Investor Transactions and Portfolios ---
num_investors = 500
investor_ids = [f"INV_{i:04d}" for i in range(1, num_investors + 1)]
cohort_years = np.random.choice([2021, 2022, 2023, 2024, 2025], size=num_investors, p=[0.1, 0.2, 0.3, 0.2, 0.2])

tx_records = []
for inv_id, c_year in zip(investor_ids, cohort_years):
    num_tx = np.random.randint(3, 15)
    base_date = datetime(c_year, 1, 1)
    
    current_date = base_date
    sip_amount = np.random.choice([2000, 5000, 10000, 15000, 25000], p=[0.3, 0.4, 0.15, 0.1, 0.05])
    chosen_fund = np.random.choice(fund_names)
    
    for _ in range(num_tx):
        gap = np.random.choice([28, 30, 31, 32, 45], p=[0.2, 0.5, 0.1, 0.1, 0.1])
        current_date += timedelta(days=int(gap))
        if current_date > datetime(2026, 1, 1):
            break
        tx_records.append({
            'investor_id': inv_id,
            'cohort_year': c_year,
            'tx_date': current_date,
            'amount': sip_amount,
            'fund_name': chosen_fund
        })

tx_df = pd.DataFrame(tx_records)

# --- 4. Sector Allocation Data for Equity Funds (Top 15) ---
sectors = ['Tech', 'Finance', 'Energy', 'Healthcare', 'Consumer Goods']
sector_allocations = []
for fund in fund_names[:15]:
    weights = np.random.dirichlet(np.ones(5))
    for sector, weight in zip(sectors, weights):
        sector_allocations.append({'fund_name': fund, 'sector': sector, 'weight': weight})
sector_df = pd.DataFrame(sector_allocations)

print("Setup successful.")


# ==============================================================================
# STEP 2: CORE ANALYTICS ENGINE AND DELIVERABLES GENERATION
# ==============================================================================

# --- DELIVERABLE 1 and 2: Historical VaR and CVaR Report ---
var_95_series = returns_df.quantile(0.05)
cvar_list = []

for fund in returns_df.columns:
    fund_returns = returns_df[fund]
    var_threshold = var_95_series[fund]
    cvar_val = fund_returns[fund_returns <= var_threshold].mean()
    cvar_list.append(cvar_val)

var_cvar_df = pd.DataFrame({
    'Fund Name': returns_df.columns,
    'Historical_VaR_95': var_95_series.values,
    'Conditional_VaR_95': cvar_list
}).set_index('Fund Name')

var_cvar_report = var_cvar_df.merge(funds_df, left_index=True, right_on='fund_name').rename(columns={'fund_name': 'Fund Name'})
var_cvar_report.to_csv('var_cvar_report.csv', index=False)
print("Saved var_cvar_report.csv")


# --- DELIVERABLE 3: Rolling 90-Day Sharpe Ratio and Chart ---
rolling_mean = returns_df.rolling(window=90).mean()
rolling_std = returns_df.rolling(window=90).std()
rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(252)

key_funds = [fund_names[0], fund_names[5], fund_names[15], fund_names[20], fund_names[30]]

plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")
for fund in key_funds:
    plt.plot(rolling_sharpe.index, rolling_sharpe[fund], label=fund, linewidth=2)

plt.title('Rolling 90-Day Sharpe Ratio Over Time', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Timeline', fontsize=12)
plt.ylabel('Annualized Sharpe Ratio', fontsize=12)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
plt.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()
plt.savefig('rolling_sharpe_chart.png', dpi=300)
plt.close()
print("Saved rolling_sharpe_chart.png")


# --- CORE PIPELINE: Investor Cohort Analysis ---
cohort_summary = tx_df.groupby('cohort_year').agg(
    avg_sip_amount=('amount', 'mean'),
    total_invested=('amount', 'sum')
).reset_index()

top_funds = tx_df.groupby(['cohort_year', 'fund_name']).size().reset_index(name='count')
top_funds = top_funds.sort_values(['cohort_year', 'count'], ascending=[True, False]).drop_duplicates('cohort_year')

cohort_analysis_df = cohort_summary.merge(top_funds[['cohort_year', 'fund_name']], on='cohort_year')
cohort_analysis_df.rename(columns={'fund_name': 'top_fund_preference'}, inplace=True)


# --- CORE PIPELINE: SIP Continuity Analysis ---
tx_df_sorted = tx_df.sort_values(['investor_id', 'tx_date'])
tx_df_sorted['prev_tx_date'] = tx_df_sorted.groupby('investor_id')['tx_date'].shift(1)
tx_df_sorted['gap_days'] = (tx_df_sorted['tx_date'] - tx_df_sorted['prev_tx_date']).dt.days

investor_counts = tx_df_sorted.groupby('investor_id').size()
eligible_investors = investor_counts[investor_counts >= 6].index

continuity_df = tx_df_sorted[tx_df_sorted['investor_id'].isin(eligible_investors)].groupby('investor_id').agg(
    avg_gap=('gap_days', 'mean'),
    max_gap=('gap_days', 'max')
).reset_index()

continuity_df['status'] = np.where(continuity_df['max_gap'] > 35, 'at-risk', 'active')


# --- CORE PIPELINE: Sector HHI Concentration ---
sector_df['weight_sq'] = sector_df['weight'] ** 2
hhi_df = sector_df.groupby('fund_name')['weight_sq'].sum().reset_index()
hhi_df.rename(columns={'weight_sq': 'HHI'}, inplace=True)
hhi_df = hhi_df.merge(funds_df[['fund_name', 'risk_grade']], on='fund_name').sort_values(by='HHI', ascending=False)


# --- DELIVERABLE 4: Fund Recommender Script ---
latest_sharpe = rolling_sharpe.iloc[-1].to_frame(name='latest_sharpe')
recommender_base = latest_sharpe.merge(funds_df, left_index=True, right_on='fund_name')

embedded_data = recommender_base[['fund_name', 'risk_grade', 'latest_sharpe']].to_dict(orient='list')

recommender_script = """
import pandas as pd

data = %s
df_rec = pd.DataFrame(data)

def get_recommendation(risk_appetite):
    risk_appetite = risk_appetite.strip().capitalize()
    if risk_appetite not in ['Low', 'Moderate', 'High']:
        print("Invalid input. Select from: Low, Moderate, High")
        return None
        
    filtered = df_rec[df_rec['risk_grade'] == risk_appetite]
    top_3 = filtered.sort_values(by='latest_sharpe', ascending=False).head(3).copy()
    top_3.columns = ['Fund Name', 'Risk Profile', 'Latest Sharpe Ratio']
    
    print(f"\\n=== TOP 3 RECOMMENDATIONS FOR {risk_appetite.upper()} RISK PROFILE ===")
    print(top_3.to_string(index=False))
    return top_3

if __name__ == '__main__':
    user_input = input("Enter your Risk Appetite (Low / Moderate / High): ")
    get_recommendation(user_input)
""" % (str(embedded_data))

with open('recommender.py', 'w', encoding='utf-8') as f:
    f.write(recommender_script.strip())
print("Saved recommender.py")


# --- VERIFICATION OUTCOME ---
def run_embedded_recommender(risk_appetite):
    filtered = recommender_base[recommender_base['risk_grade'] == risk_appetite]
    top_3 = filtered.sort_values(by='latest_sharpe', ascending=False).head(3).copy()
    top_3 = top_3[['fund_name', 'risk_grade', 'latest_sharpe']].rename(
        columns={'fund_name': 'Fund Name', 'risk_grade': 'Risk Profile', 'latest_sharpe': 'Sharpe Ratio'}
    )
    return top_3

print("\n--- SAMPLE INTERACTIVE RECOMMENDER OUTPUT (MODERATE RISK) ---")
print(run_embedded_recommender('Moderate').to_markdown(index=False))

Setup successful.
Saved var_cvar_report.csv
Saved rolling_sharpe_chart.png
Saved recommender.py

--- SAMPLE INTERACTIVE RECOMMENDER OUTPUT (MODERATE RISK) ---
| Fund Name             | Risk Profile   |   Sharpe Ratio |
|:----------------------|:---------------|---------------:|
| Beta Steady Growth 10 | Moderate       |        5.7096  |
| Beta Steady Growth 1  | Moderate       |        2.87251 |
| Beta Steady Growth 13 | Moderate       |        1.92625 |
